In [1]:
!pip install ultralytics -q

import os
import glob
import random
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO
from IPython.display import display, Image


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
YAML_PATH="/kaggle/input/datasets/roronazoromani/ppe-filtered/ppe_filtered/data.yaml"

print(YAML_PATH)

# ============================================================
# LOAD MODEL
# ============================================================

model=YOLO("yolov8s.pt")


/kaggle/input/datasets/roronazoromani/ppe-filtered/ppe_filtered/data.yaml


In [3]:
results=model.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=768,
    batch=16,
    device=0,
    workers=8,
    optimizer="AdamW",
    lr0=0.001,
    patience=20,
    pretrained=True,
    cache=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=2,
    perspective=0.0005,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    plots=True,
    verbose=True,
    project="ppe_training",
    name="filtered_ppe_yolov8s"
)

print("\nTRAINING COMPLETED")


Ultralytics 8.4.51 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/roronazoromani/ppe-filtered/ppe_filtered/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.2, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=filtered_ppe_yolov8s, nbs=64, nms=False, opset=None, optimize=

In [4]:
# ============================================================
# BEST MODEL PATH
# ============================================================

best_model_path="/kaggle/working/ppe_training/filtered_ppe_yolov8s/weights/best.pt"

print("\nBEST MODEL:")
print(best_model_path)

# ============================================================
# LOAD BEST MODEL
# ============================================================

best_model=YOLO(best_model_path)



BEST MODEL:
/kaggle/working/ppe_training/filtered_ppe_yolov8s/weights/best.pt


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/ppe_training/filtered_ppe_yolov8s/weights/best.pt'

In [ ]:
metrics=best_model.val()

print("\n========== FINAL METRICS ==========")

print("mAP50:",metrics.box.map50)

print("mAP50-95:",metrics.box.map)

print("Precision:",metrics.box.mp)

print("Recall:",metrics.box.mr)


In [ ]:
results_dir="/kaggle/working/ppe_training/filtered_ppe_yolov8s"

graphs=[
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
    "PR_curve.png"
]

print("\nDISPLAYING TRAINING GRAPHS")

for graph in graphs:
    graph_path=os.path.join(results_dir,graph)
    if os.path.exists(graph_path):
        print(graph)
        display(Image(filename=graph_path))


In [ ]:
results_csv=os.path.join(results_dir,"results.csv")

df=pd.read_csv(results_csv)

print("\nRESULTS CSV")

print(df.head())


In [ ]:
# ============================================================
# LOSS CURVE
# ============================================================

plt.figure(figsize=(12,6))

plt.plot(
    df["epoch"],
    df["train/box_loss"],
    label="Train Box Loss"
)

plt.plot(
    df["epoch"],
    df["val/box_loss"],
    label="Validation Box Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("Box Loss Curve")

plt.legend()

plt.show()

# ============================================================
# mAP CURVE
# ============================================================

plt.figure(figsize=(12,6))

plt.plot(
    df["epoch"],
    df["metrics/mAP50(B)"],
    label="mAP50"
)

plt.plot(
    df["epoch"],
    df["metrics/mAP50-95(B)"],
    label="mAP50-95"
)

plt.xlabel("Epoch")

plt.ylabel("mAP")

plt.title("mAP Curve")

plt.legend()

plt.show()

# ============================================================
# PRECISION RECALL CURVE
# ============================================================

plt.figure(figsize=(12,6))

plt.plot(
    df["epoch"],
    df["metrics/precision(B)"],
    label="Precision"
)

plt.plot(
    df["epoch"],
    df["metrics/recall(B)"],
    label="Recall"
)

plt.xlabel("Epoch")

plt.ylabel("Score")

plt.title("Precision Recall Curve")

plt.legend()

plt.show()


In [ ]:
# ============================================================
# TEST INFERENCE
# ============================================================

test_images=glob.glob(
    "/kaggle/working/ppe_filtered/test/images/*"
)

sample_images=random.sample(test_images,5)

print("\nRUNNING TEST INFERENCE")

for img_path in sample_images:

    best_model.predict(

        source=img_path,

        imgsz=768,

        conf=0.25,

        save=True
    )

# ============================================================
# DISPLAY PREDICTIONS
# ============================================================

pred_path="/kaggle/runs/detect/predict"

pred_images=glob.glob(f"{pred_path}/*.jpg")

print("\nSAMPLE PREDICTIONS")

for img in pred_images[:5]:

    display(Image(filename=img))

print("\nTRAINING PIPELINE COMPLETED SUCCESSFULLY")